# DMOSpeech 2 - EXP-035 + EXP-036 All-in-One: Path A Invariance Ladder & Unseen SNR_R + Path B Novel Mechanisms

- **Task ID:** `EVAL-030` (combined prospective T4 sweep on `unseen_cal[30:60]`, strictly disjoint from all 100 development speakers `EXP-001..034` and EXP-028 `unseen_cal[0:30]`).
- **Why combined in 1 run:** Shares the expensive baselines (`STU`, `STUR2`, `CPH8`, `CPQH`, `P25`, and Whisper/WavLM/UTMOS scoring) on identical seeds, cutting total GPU hours nearly in half while giving paired contrasts across both Path A and Path B.
- **Path A (Condition-Invariance Ladder & Unseen Confirmation):**
  - Tests `Base CPH8` (`12`) -> `CPH8_DP` (`13`) -> `FCH_DP` (`13`) -> `G1_VSWAP` (`13`) -> `G2_AFFINE` (`13`) -> `CPQH` (`13`, $G_3$) at commit point `t = 0.2929, j = 2` with draft prompt frames locked.
  - Evaluates `SNR_R`, `SNR_ONLY`, `SNR_DRIFT`, `R_ONLY`, `W8`, and 5-fold speaker-grouped OOF calibration.
- **Path B (Novel Inference-Time Mechanisms, 13 passes, NO binary `20 dB` / `0.340` gate):**
  - `DSGH_MICRO` (`macro_weight=0.0`) & `DSGH_DUAL` (`macro_weight=0.5`): Continuous 1D Wasserstein-2 quantile soft-shrinkage + excess-roughness rank-coherence damping toward `s0`.
  - `CWPS`: Closed-Loop Wasserstein-Proximal Teacher Sampling running `s0` first (`1` pass), projecting CFG Euler steps `t_1..t_3` along the W2 geodesic (`10` teacher passes), and finishing at `j=2` (`2` student passes).
- **Cases:** `240` quality cases (`unseen_cal[30:60]` x 8 conditions), `40` diversity blocks (`40:60` x 12 seeds x `{clean, gauss25}`), `10` clean timing cases (`30:40`).
- **Estimated T4 runtime:** ~4.5 hours with `STAGE = "all"` (continuously synced every 60s to Drive: if disconnected, just **Run all** again to resume).
- **Outputs:** Synced to `MyDrive/exp035_036_all_in_one/` and exported as `DMOSpeech2_EXP035_036_AllInOne_Results.zip`.

In [ ]:
import os
import subprocess

# GPU checked with nvidia-smi, not PyTorch: the notebook kernel then holds no PyTorch or CUDA state in RAM,
# so the runner (two TTS models, Whisper large-v3, WavLM and UTMOS) has all of it. Results are unaffected.
q = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), "No GPU found. Switch the runtime to T4 GPU (Runtime > Change runtime type)."
gpu_name = q.stdout.strip().splitlines()[0]
print("GPU:", gpu_name)
assert "T4" in gpu_name, f"Expected a Tesla T4 (found {gpu_name}): the smoke test matches the thesis Colab Tesla T4 baseline."

from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/results"
DRIVE_DIR = "/content/drive/MyDrive/exp035_036_all_in_one"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("Drive directories ready:", DRIVE_DIR)

## 1. Clone Repositories, Install Dependencies & Verify Preregistration

Clones `Shonget-DMOSpeech2` (pinned commit `8a7868fc...`) and `Final-Thesis-Setup` (branch `rhnl-3`), which holds `code/colab/run_exp035_036_combined.py` and `research/experiments/EXP-035-036-all-in-one/preregistration.json`.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_BRANCH = "rhnl-3"
DMO_COMMIT = "8a7868fc3345702a586173447f4869205b501f26"
DMO_REPO = "https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git"
THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
STAGE = "all"  # runs quality (240 cases), then variety (20 speakers x 2 conditions x 12 seeds), then timing (10 cases); resumes from Drive. Other options: "quality", "diversity" or "timing"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
thesis_url = f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git"

if not os.path.exists(DMO_ROOT):
    subprocess.run(["git", "clone", DMO_REPO, DMO_ROOT], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "checkout", "-q", DMO_COMMIT], check=True)

if not os.path.exists(THESIS_ROOT):
    subprocess.run(["git", "clone", "-b", THESIS_BRANCH, thesis_url, THESIS_ROOT], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "fetch", "-q", "origin", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "checkout", "-q", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "pull", "-q", "origin", THESIS_BRANCH], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "numpy==2.2.3", "pandas==2.2.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "-r", f"{DMO_ROOT}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q",
    "torchdiffeq", "vocos", "speechbrain", "transformers", "jiwer",
    "soundfile", "librosa", "pyarrow", "scipy", "jieba", "accelerate", "huggingface_hub"
], check=True)
import gc, shutil; shutil.rmtree("/root/.cache/pip", ignore_errors=True); gc.collect()

prereg_path = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/preregistration.json"
assert os.path.exists(prereg_path), f"Missing {prereg_path} on branch {THESIS_BRANCH}"
prereg = json.loads(open(prereg_path, encoding="utf-8").read())
print(f"Preregistration verified: sha256={prereg['sha256'][:12]}..., qual={prereg['n_qual_cases']}, div={prereg['n_div_speakers']}x{prereg['div_seeds']}")

## 2. Checkpoints & Seed-TTS English Audio (Drive Cache Supported)


In [ ]:
import hashlib
import os
import shutil
import subprocess

DMO_ROOT = "/content/Shonget-DMOSpeech2"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(f"{DMO_ROOT}/ckpts", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/ckpts", exist_ok=True)

CKPTS = {
    "model_85000.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt",
        "e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c",
    ),
    "model_1500.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt",
        "6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2",
    ),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, expected) in CKPTS.items():
    local_p = f"{DMO_ROOT}/ckpts/{name}"
    cache_p = f"{DRIVE_CACHE}/ckpts/{name}"
    if not os.path.exists(local_p) and os.path.exists(cache_p):
        shutil.copy2(cache_p, local_p)
    if not os.path.exists(local_p):
        subprocess.run(["wget", "-c", url, "-O", local_p], check=True)
    digest = sha256_file(local_p)
    assert digest == expected, f"{name} SHA-256 mismatch: got {digest}"
    if not os.path.exists(cache_p):
        shutil.copy2(local_p, cache_p)
    print(f"{name}: SHA-256 verified")

data_dir = f"{DMO_ROOT}/data/seed-tts-eval"
probe_wav = f"{data_dir}/en/prompt-wavs/common_voice_en_10933823.wav"
cache_data = f"{DRIVE_CACHE}/seed-tts-eval"
if (not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000) and os.path.exists(f"{cache_data}/en/meta.lst"):
    shutil.copytree(cache_data, data_dir, dirs_exist_ok=True)
if not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000:
    from huggingface_hub import snapshot_download
    shutil.rmtree(data_dir, ignore_errors=True)
    snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir=data_dir)
assert os.path.exists(probe_wav) and os.path.getsize(probe_wav) >= 1000, "Seed-TTS audio missing"
print("Seed-TTS audio verified [OK]")

## 3. Cafeteria Babble Noise (DEMAND, Drive Cache Supported)

The `babble15` condition mixes DEMAND's cafeteria recording (`PCAFETER/ch01.wav`, Thiemann, Ito and Vincent, 2013) into the prompt at 15 dB, as in EXP-011 and EXP-031. The archive is cached in `MyDrive/dmo_cache/demand/` after the first download.

In [ ]:
import hashlib
import os
import shutil
import subprocess

DEMAND = "/content/demand"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
ZIP_URL = "https://zenodo.org/records/1227121/files/PCAFETER_16k.zip?download=1"
LOCAL_ZIP = "/content/PCAFETER_16k.zip"
CACHE_ZIP = f"{DRIVE_CACHE}/demand/PCAFETER_16k.zip"
WAV = f"{DEMAND}/PCAFETER/ch01.wav"

if not os.path.exists(WAV):
    os.makedirs(DEMAND, exist_ok=True)
    if not os.path.exists(LOCAL_ZIP) and os.path.exists(CACHE_ZIP):
        shutil.copy2(CACHE_ZIP, LOCAL_ZIP)
    if not os.path.exists(LOCAL_ZIP):
        subprocess.run(["wget", "-q", "-c", "-O", LOCAL_ZIP, ZIP_URL], check=True)
    subprocess.run(["unzip", "-q", "-o", LOCAL_ZIP, "-d", DEMAND], check=True)
    if not os.path.exists(CACHE_ZIP):
        os.makedirs(os.path.dirname(CACHE_ZIP), exist_ok=True)
        shutil.copy2(LOCAL_ZIP, CACHE_ZIP)
if not os.path.exists(WAV):
    print(subprocess.run(["find", DEMAND, "-maxdepth", "3"], capture_output=True, text=True).stdout[:2000])
    raise FileNotFoundError("PCAFETER/ch01.wav not found: check the archive layout printed above")
with open(WAV, "rb") as f:
    digest = hashlib.sha256(f.read()).hexdigest()
print(f"DEMAND babble track ready: {WAV} ({os.path.getsize(WAV)} bytes, sha256 {digest[:12]}...)")


## 4. Restore Any Partial Results from Google Drive


In [ ]:
import csv
import os
import shutil

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/results"
DRIVE_DIR = "/content/drive/MyDrive/exp035_036_all_in_one"
os.makedirs(OUT_DIR, exist_ok=True)

for rel in ("quality.csv", "diversity.csv", "timing.csv", "smoke/quality.csv", "smoke/diversity.csv",
            "smoke/timing.csv", "smoke/smoke_check.json"):
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(OUT_DIR, rel)
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Restored from Drive:", rel)

for fname, target in (("quality.csv", 240), ("diversity.csv", 40), ("timing.csv", 10)):
    p = os.path.join(OUT_DIR, fname)
    n = sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
    print(f"{fname}: {n} / {target} rows")


## 5. Pre-Flight Smoke Test

Verifies preregistration hashes, G1 shift invariance, G2 affine invariance, FCH floor invariance, `DSGH` self-identity (`T == S`), exact pass counts across all 12 arms (`PassCounter`, including `CWPS`), and standalone-vs-shared parity on `unseen_cal[30]`, then runs micro stages.

If it stops with exit code -9 (`died with <Signals.SIGKILL: 9>`), the machine ran out of RAM: use **Runtime > Disconnect and delete runtime**, then **Run all**.

In [ ]:
import json
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_exp035_036_combined.py"
DEMAND = "/content/demand"

def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {
    "MALLOC_ARENA_MAX": "2", "MALLOC_MMAP_THRESHOLD_": "65536",
    "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
    "TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "PYTHONUNBUFFERED": "1",
}

os.chdir(THESIS_ROOT)
proc = subprocess.Popen([sys.executable, "-u", RUNNER, "smoke", "--dmo_root", DMO_ROOT, "--demand_dir", DEMAND, "--out_dir", OUT_DIR],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
if code in (-9, 137):
    raise SystemExit("The smoke test was killed (exit code -9): the machine ran out of RAM. "
                     "Use Runtime > Disconnect and delete runtime, then Run all again.")
assert code == 0, f"Smoke test failed with code {code}"
print(json.dumps(json.loads(open(f"{OUT_DIR}/smoke/smoke_check.json", encoding="utf-8").read()), indent=2))

## 6. Run Synthesis (`all` = about 4.5 hours: quality ~1 h 45 min, variety ~1 h 30 min, timing ~10 min)

In [ ]:
import os
import shutil
import subprocess
import sys
import threading

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/results"
DRIVE_DIR = "/content/drive/MyDrive/exp035_036_all_in_one"
RUNNER = f"{THESIS_ROOT}/code/colab/run_exp035_036_combined.py"
DEMAND = "/content/demand"
STAGE = str(globals().get("STAGE", "all")).strip().lower()  # set STAGE in the setup cell (Step 1), not here
assert STAGE in ("quality", "diversity", "timing", "all"), f"STAGE must be 'quality', 'diversity', 'timing' or 'all', not {STAGE!r}"
print("Stage:", STAGE, {"all": "(about 4.5 hours)", "quality": "(about 1 h 45 min)", "diversity": "(about 1 h 30 min)", "timing": "(about 10 min)"}[STAGE])

def sync_to_drive():
    if not os.path.exists(OUT_DIR):
        return
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            src = os.path.join(root, fn)
            if not os.path.exists(src) or fn.startswith("tmp_"):
                continue
            rel = os.path.relpath(src, OUT_DIR)
            dst = os.path.join(DRIVE_DIR, rel)
            if (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)
                    and os.path.getmtime(dst) >= os.path.getmtime(src)):
                continue  # unchanged since the last sync
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)

stop_evt = threading.Event()
def sync_loop():
    while not stop_evt.wait(60):
        try:
            sync_to_drive()
        except Exception as exc:
            print("Drive sync warning:", exc)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {
    "MALLOC_ARENA_MAX": "2", "MALLOC_MMAP_THRESHOLD_": "65536",
    "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
    "TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "PYTHONUNBUFFERED": "1",
}

os.chdir(THESIS_ROOT)
code = None
try:
    proc = subprocess.Popen([sys.executable, "-u", RUNNER, "run", "--dmo_root", DMO_ROOT, "--demand_dir", DEMAND, "--out_dir", OUT_DIR, "--stage", STAGE],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
finally:
    stop_evt.set()
    t.join(timeout=5)
    sync_to_drive()
if code in (-9, 137):
    raise SystemExit("The run was killed (exit code -9): the machine ran out of RAM. Use Runtime > Disconnect and delete runtime, "
                     "then Run all: Step 3 restores the finished cases from Drive and the run resumes from them.")
assert code == 0, f"Run failed with code {code}"
print("Run finished and synced to Drive.")

## 7. Analyze & Export Results ZIP

Runs the full analysis once `quality.csv` (240 rows) and `diversity.csv` (40 rows) are complete, otherwise a partial one. The results zip (CSVs, report and manifest) is saved to Drive first, then downloaded; if the browser tab has lost its connection, take it from Drive.

In [ ]:
import csv
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import files

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-035-036-all-in-one/results"
DRIVE_DIR = "/content/drive/MyDrive/exp035_036_all_in_one"
RUNNER = f"{THESIS_ROOT}/code/colab/run_exp035_036_combined.py"
ZIP_NAME = "DMOSpeech2_EXP035_036_AllInOne_Results.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

def n_rows(name):
    p = f"{OUT_DIR}/{name}"
    return sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0

cmd = [sys.executable, RUNNER, "analyze", "--out_dir", OUT_DIR]
if n_rows("quality.csv") < 240 or n_rows("diversity.csv") < 40:
    cmd.append("--allow_partial")
if n_rows("timing.csv") < 10:
    print("Note: timing.csv is incomplete; the cost table will cover the finished cases only.")
os.chdir(THESIS_ROOT)
subprocess.run(cmd, check=True)
print(open(f"{OUT_DIR}/EXP035_036_COMBINED_REPORT.md", encoding="utf-8").read())

for root, _, fns in os.walk(OUT_DIR):
    for fn in fns:
        src = os.path.join(root, fn)
        if fn.startswith("tmp_"):
            continue
        rel = os.path.relpath(src, OUT_DIR)
        dst = os.path.join(DRIVE_DIR, rel)
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src) and os.path.getmtime(dst) >= os.path.getmtime(src):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(OUT_DIR):
        for fn in fns:
            src = os.path.join(root, fn)
            if fn.startswith("tmp_") or fn.endswith((".wav", ".npz")):
                continue
            zf.write(src, arcname=os.path.relpath(src, OUT_DIR))
shutil.copy2(ZIP_PATH, os.path.join(DRIVE_DIR, ZIP_NAME))
print("Packaged:", ZIP_PATH)
print("Saved to Drive:", os.path.join(DRIVE_DIR, ZIP_NAME))

files.download(ZIP_PATH)
